Load the file and check the data making sure there are not a lot of null values

In [96]:
import pandas as pd
df = pd.read_json("../data/raw/dramas.jsonl", lines=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             5000 non-null   int64         
 1   url            5000 non-null   str           
 2   titles         5000 non-null   object        
 3   cover          4990 non-null   str           
 4   synopsis       5000 non-null   str           
 5   score          5000 non-null   object        
 6   country        5000 non-null   str           
 7   type           5000 non-null   str           
 8   date           4999 non-null   datetime64[us]
 9   episodes       5000 non-null   int64         
 10  duration       5000 non-null   str           
 11  network        5000 non-null   object        
 12  genres         5000 non-null   object        
 13  tags           5000 non-null   object        
 14  rating         2892 non-null   str           
 15  directors      5000 non-null   o

In [97]:
df.isnull().sum()

id                  0
url                 0
titles              0
cover              10
synopsis            0
score               0
country             0
type                0
date                1
episodes            0
duration            0
network             0
genres              0
tags                0
rating           2108
directors           0
screenwriters       0
cast                0
dtype: int64

In [105]:
df = pd.read_json("../data/raw/tvshows.jsonl", lines=True)
df["country"].value_counts()

country
South Korea    2621
China           940
Japan           425
Thailand        241
Philippines      47
Hong Kong        43
Taiwan           15
Name: count, dtype: int64

In [106]:
paths = {
    "drama": "../data/raw/dramas.jsonl",
    "movie": "../data/raw/movies.jsonl",
    "tvshow": "../data/raw/tvshows.jsonl",
    "special": "../data/raw/specials.jsonl",
}

dfs = []
for content_type, path in paths.items():
    d = pd.read_json(path, lines=True)
    d["content_type"] = content_type
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True)

Clean the data to be Chinese shows only and grouping all content types into a single dataFrame

In [107]:
df = df[df["country"] == "China"].copy()

Checking whether there are duplicates title and we should check for duplicates by ID

In [108]:
df["title_str"] = df["titles"].apply(lambda t: t.get("english") or t.get("native"))
dupe_titles = df[df.duplicated(subset="title_str", keep=False)]
dupe_titles[["title_str", "content_type", "id"]].sort_values("title_str")

,title_str,content_type,id
3729,Be Yourself,drama,732397
907,Be Yourself,drama,65847
1337,Be with You,drama,49155
10353,Be with You,tvshow,696527
1369,Begin Again,drama,767927
...,...,...,...
4088,You Complete Me,drama,750833
3668,You Complete Me,drama,55061
6160,Youth,movie,22091
1712,Youth,drama,24642


In [109]:
import pandas as pd
df = pd.read_pickle("../data/processed/clean_dramas.pkl")
print(df["soup"].iloc[0])
print(df["content_type"].value_counts())

Title: Nirvana in Fire
Type: drama
Genres: Military, Historical, Drama, Political
Tags: Power Struggle, Smart Male Lead, Scheme, Death, War, Hidden Identity, Sibling Rivalry, Corruption, Political Intrigue, Revenge
Cast: Hu Ge, Liu Tao, Wang Kai, Victor Huang, Chen Long, Ding Yong Dai, Liu Min Tao
Director: Kong Sheng, Li Xue
Year: 2015
Description: In sixth-century China, the Emperor of Great Liang orders the unjust execution of his brother-in-law Marshal Lin Xie alongside the Lin family, his 70,000 army soldiers, and Crown Prince Qi. Secretly surviving the massacre is Lin Xie's son, Lin Shu, who undergoes medical treatment that changes his appearance entirely and leaves him in a weakened state, unable to ever perform martial arts again. Lin Shu changed his name to Mei Chang Su and later became the chief of the pugilist world and established the Jiangzuo Alliance.

Twelve years later, Mei Chang Su returns to the capital with a secret plan after being sought after by Prince Yu and Prin

Make sure that cosine similarity between two similar shows should be decently high

In [111]:
from sklearn.metrics.pairwise import cosine_similarity

idx_a = df[df["soup"].str.contains("Nirvana in Fire")].index[0]
idx_b = df[df["soup"].str.contains("Love Like the Galaxy")].index[0]  # or any other historical political drama

sim = cosine_similarity(embeddings[[idx_a]], embeddings[[idx_b]])
print(sim)

[[0.63953096]]


Making sure that two seperate dramas that are unrelated have low score

In [112]:
idx_c = df[df["soup"].str.contains("Hidden Love")].index[0]

sim_unrelated = cosine_similarity(embeddings[[idx_a]], embeddings[[idx_c]])
print(sim_unrelated)

[[0.48186475]]


In [117]:
import sys
sys.path.append("../src")
from recommendation.similarity import load_data, load_model, search

df, embeddings = load_data()
model = load_model()

search("modern college romance", df, embeddings, model, k=10)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6521.19it/s]


,soup,similarity,content_type
426,Title: First Romance\nType: drama\nGenres: Com...,0.588438,drama
522,Title: Binary Love\nType: drama\nGenres: Comed...,0.579004,drama
1118,Title: Nice to Meet You\nType: drama\nGenres: ...,0.573493,drama
417,Title: A Female Student Arrives at the Imperia...,0.572117,drama
268,Title: Be Yourself\nType: drama\nGenres: Comed...,0.547973,drama
1139,Title: Unrequited Love\nType: drama\nGenres: R...,0.547714,drama
313,Title: Please Classmate\nType: drama\nGenres: ...,0.543532,drama
438,Title: Make My Heart Smile\nType: drama\nGenre...,0.543366,drama
1875,Title: The Raccoon\nType: movie\nGenres: Roman...,0.541262,movie
1451,Title: Guyuan Imperial College\nType: drama\nG...,0.539694,drama


In [123]:
import importlib
import data.mdl_client
importlib.reload(data.mdl_client)
from data.mdl_client import search_title, get_slug, get_recommendations, build_relevant_list


slug = get_slug("Nirvana in Fire")
print(slug)

9025-nirvana-in-fire


In [124]:
recs = get_recommendations(slug)
print(recs)

{'recommendations': [{'title': 'Six Flying Dragons', 'year': '2015', 'slug': '14324-six-flying-dragons', 'url': 'https://mydramalist.com/14324-six-flying-dragons', 'image': 'https://i.mydramalist.com/bBvWjt.jpg', 'rating': '8.6', 'reasons': ['Both are the stories of power play, entanglement into politics and scheming which are inspired by the Asian history. Both center around secret organizations and backroom deals, lofty ideals and seeking for justice.', 'However, the characters of NiF are absolutely fictional, it has something of fantasy ,  it shows how justice prevails at the end whereas SFL aspires to some real historicity and believability (though there are also flying martial artists there), its characters are more realistic and multi-dimensional.', 'Also NiF looks more expensive and the plot is a little bit more suspensefull and accelerated, imho.'], 'recommended_by': 'An S', 'votes': '18'}, {'title': 'The Disguiser', 'year': '2015', 'slug': '13061-the-disguiser', 'url': 'https:

In [125]:
test = build_relevant_list("Nirvana in Fire", df)
print(test)

['The Disguiser', 'The Rise of Phoenixes', 'Joy of Life', 'Legend of Zang Hai', "The Longest Day in Chang'an", 'Battle of Changsha', 'Nirvana in Fire Season 2: The Wind Blows in Chang Lin', 'Mysterious Lotus Casebook', 'The Double', 'The Ingenious One', 'The Legend of Anle', 'Love Like the Galaxy: Part 2', 'The Story of Ming Lan', 'Three Kingdoms', 'One and Only', 'The Untamed', 'Love in Between', 'The Vigilantes in Masks', 'Joy of Life Season 2', 'Scarlet Heart', 'Who Rules the World', 'The Rebel Princess', 'Legend of Fu Yao', 'The Sleuth of Ming Dynasty', 'The Blood of Youth', 'The Moon Brightens for You', 'The Warring States', 'The Wolf', 'Ever Night', 'Royal Nirvana', 'Tribes and Empires: Storm of Prophecy', 'Sword Snow Stride', 'The Princess Wei Young', 'Story of Yanxi Palace', 'The Advisors Alliance', 'Destined']


Set of draft_qeuries

In [126]:
draft_queries = [
    {"query": "historical romance with political intrigue", "seed_title": "Nirvana in Fire"},
    {"query": "wuxia revenge story", "seed_title": "The Legend of the Condor Heroes"},
    {"query": "modern college romance", "seed_title": "A Love So Beautiful"},
    {"query": "workplace romance with lots of comedy", "seed_title": "Love O2O"},
    {"query": "Chinese crime mystery", "seed_title": "The Long Night"},
    {"query": "family drama about complicated relationships between siblings", "seed_title": "Go Ahead"},
    {"query": "fantasy romance with gods and immortals", "seed_title": "Eternal Love"},
    {"query": "coming-of-age story about friendship and first love", "seed_title": "When We Were Young"},
    {"query": "revenge thriller", "seed_title": "The Double"},
    {"query": "quiet slice-of-life drama about ordinary people", "seed_title": "Minning Town"},
    {"query": "historical court drama with power struggles", "seed_title": "The Longest Day in Chang'an"},
    {"query": "martial arts adventure", "seed_title": "Mysterious Lotus Casebook"},
    {"query": "sweet modern romance", "seed_title": "Hidden Love"},
    {"query": "office romance between coworkers who slowly fall in love", "seed_title": "The Rational Life"},
    {"query": "detective mystery with a dark atmosphere", "seed_title": "Under the Skin"},
    {"query": "modern family drama about relationships, careers, and friendship", "seed_title": "Ode to Joy"},
    {"query": "xianxia fantasy with tragic romance", "seed_title": "Love Between Fairy and Devil"},
    {"query": "high school friendship", "seed_title": "With You"},
    {"query": "historical revenge and political conspiracy", "seed_title": "The Rise of Phoenixes"},
    {"query": "small-town slice of life with romance", "seed_title": "Meet Yourself"},
    {"query": "wuxia mystery with a group of unlikely heroes", "seed_title": "Side Story of Fox Volant"},
    {"query": "romantic comedy about two people reconnecting after school", "seed_title": "You Are My Glory"},
    {"query": "intense crime thriller about a serial killer", "seed_title": "Burning Ice"},
    {"query": "sweet high school romance with friendship and youthful coming-of-age", "seed_title": "When I Fly Towards You"},
    {"query": "historical drama about rival kingdoms, military strategy, and complicated loyalties", "seed_title": "The Long Ballad"},
    {"query": "fantasy adventure with demons, magic, and a slow-burn romance", "seed_title": "The Untamed"},
    {"query": "emotional family story centered on three unrelated children growing up together", "seed_title": "The Bond"},
    {"query": "dark mystery where the investigation gradually uncovers a much larger conspiracy", "seed_title": "The Bad Kids"},
    {"query": "lighthearted romance with an awkward but lovable male lead", "seed_title": "Put Your Head on My Shoulder"},
    {"query": "romantic comedy about two people from very different backgrounds", "seed_title": "My Little Happiness"},
]

In [127]:
missing = []
for item in draft_queries:
    match = df[df["soup"].str.contains(f"Title: {item['seed_title']}\n", regex=False)]
    if match.empty:
        missing.append(item["seed_title"])

print(f"{len(missing)} missing out of {len(draft_queries)}")
print(missing)

0 missing out of 30
[]


In [128]:
import time
import json

all_queries = []
for item in draft_queries:
    try:
        relevant = build_relevant_list(item["seed_title"], df)
    except Exception as e:
        print(f"Failed for {item['seed_title']}: {e}")
        relevant = []
    all_queries.append({
        "query": item["query"],
        "seed_title": item["seed_title"],
        "relevant": relevant
    })
    time.sleep(1)

with open("../data/processed/queries.json", "w") as f:
    json.dump(all_queries, f, indent=2, ensure_ascii=False)

print(f"Saved {len(all_queries)} queries")

Saved 30 queries


In [129]:
for item in all_queries:
    print(f"{item['query']}: {len(item['relevant'])} relevant titles")

historical romance with political intrigue: 36 relevant titles
wuxia revenge story: 4 relevant titles
modern college romance: 2 relevant titles
workplace romance with lots of comedy: 40 relevant titles
Chinese crime mystery: 4 relevant titles
family drama about complicated relationships between siblings: 19 relevant titles
fantasy romance with gods and immortals: 0 relevant titles
coming-of-age story about friendship and first love: 0 relevant titles
revenge thriller: 33 relevant titles
quiet slice-of-life drama about ordinary people: 0 relevant titles
historical court drama with power struggles: 5 relevant titles
martial arts adventure: 18 relevant titles
sweet modern romance: 27 relevant titles
office romance between coworkers who slowly fall in love: 8 relevant titles
detective mystery with a dark atmosphere: 10 relevant titles
modern family drama about relationships, careers, and friendship: 2 relevant titles
xianxia fantasy with tragic romance: 0 relevant titles
high school friend

In [130]:
zero_seeds = ["The Long Night", "Eternal Love", "When We Were Young", "The Double",
              "Minning Town", "Hidden Love", "The Rise of Phoenixes", "Side Story of Fox Volant",
              "The Bad Kids"]  # your actual 9 zero seeds

for seed in zero_seeds:
    slug = get_slug(seed)
    recs = get_recommendations(slug)
    print(f"{seed} -> slug={slug}, total_recs={recs.get('total')}")

The Long Night -> slug=49485-the-long-night, total_recs=10
Eternal Love -> slug=686261-eternal-love, total_recs=75
When We Were Young -> slug=21570-when-we-were-young, total_recs=8
The Double -> slug=736749-di-jia-qian-jin, total_recs=75
Minning Town -> slug=65373-minning-town, total_recs=5
Hidden Love -> slug=729705-hidden-love, total_recs=75
The Rise of Phoenixes -> slug=21032-the-rise-of-phoenixes, total_recs=43
Side Story of Fox Volant -> slug=688445-fei-hu-wai-zhuan, total_recs=15
The Bad Kids -> slug=51571-cat-s-cradle, total_recs=21


In [131]:
result = search_title("The Bad Kids")
for m in result["results"]:
    print(m["title"], "|", m["slug"], "|", m.get("year"))

The Bad Kids | 774829-the-bad-kids | 2024
The Bad Kids | 51571-cat-s-cradle | 2020
The Bad Dad | 25323-the-bad-dad | 2007
Kids' Lives Matter | 684677-kids-lives-matter | 2021
Yok Luerd Mungkorn | 11205-yok-luerd-mungkorn | 2012
Kids on the Slope | 23460-sakamichi-no-apollon | 2018
Wataru Seken wa Oni Bakari | 10886-wataru-seken-wa-oni-bakari | 1990
Kids War Special: Zaken na yo | 715743-kids-war-special-zaken-na-yo | 2002
Pirate of the Lost Sea | 58497-pirate-of-the-lost-sea | 2008
The Legacy | 5540-become-pretty-things | 2014


In [139]:
results = search("historical romance with political intrigue", df, embeddings, model, k=10)
print(results[["soup", "similarity"]].apply(lambda r: (r["soup"].split("\n")[0], r["similarity"]), axis=1).tolist())

[('Title: Twisted Fate of Love', 0.5772205591201782), ('Title: Untouchable Lovers', 0.572457492351532), ('Title: A Flower on the Continent', 0.566983699798584), ('Title: Everlasting Longing', 0.5653089284896851), ('Title: A Tale of Love and Loyalty', 0.5621240139007568), ('Title: Stolen Love', 0.5581011176109314), ('Title: Enslaved by Love', 0.5547134876251221), ('Title: The Love Duel', 0.5537896156311035), ("Title: The Princess's Gambit", 0.5501770973205566), ('Title: Blossom', 0.5472314953804016)]


In [145]:
from recommendation.llm import expand_query

for q in ["historical romance with political intrigue", "detective mystery with a dark atmosphere", "wuxia mystery with a group of unlikely heroes"]:
    print(f"ORIGINAL: {q}")
    print(f"EXPANDED: {expand_query(q)}")
    print()

ORIGINAL: historical romance with political intrigue
EXPANDED: historical romance with political intrigue

ORIGINAL: detective mystery with a dark atmosphere
EXPANDED: A brooding, rain‑slick cityscape where neon flickers over foggy alleys, populated by corrupt officials,

ORIGINAL: wuxia mystery with a group of unlikely heroes
EXPANDED: A sprawling martial‑arts saga set in mist‑shrouded Jianghu, where a ragtag band—an ex‑imperial guard haunted by a past betrayal, a clever but low‑born thief with a secret map, a reclusive monk who doubts his vows, and a spirited merchant’s daughter wielding hidden sword techniques—are thrust together by a cryptic murder at a remote



In [147]:
import importlib
import recommendation.llm
importlib.reload(recommendation.llm)
from recommendation.llm import extract_preferences

for q in ["historical romance with political intrigue", "detective mystery with a dark atmosphere", "wuxia mystery with a group of unlikely heroes"]:
    print(q, "->", extract_preferences(q))

historical romance with political intrigue -> {'genres': ['historical', 'romance'], 'themes': ['political', 'intrigue'], 'exclude': []}
detective mystery with a dark atmosphere -> {'genres': ['detective', 'mystery'], 'themes': ['dark', 'noir'], 'exclude': []}
wuxia mystery with a group of unlikely heroes -> {'genres': ['wuxia', 'mystery'], 'themes': ['unlikely heroes'], 'exclude': []}
